# Reproducing MAR-B (Autoregressive Image Generation Without Vector Quantization)

**Paper:** [arXiv:2406.11838](https://arxiv.org/abs/2406.11838) — Li, Tian, Li, Deng, He, NeurIPS 2024 Spotlight
**Repository:** [LTH14/mar](https://github.com/LTH14/mar)
**Pretrained:** MAR-B (208M params, paper FID-50K = 2.31)

## Honest scope of this run

- Hardware: Kaggle free-tier T4 (16GB VRAM, 9h session)
- Model: **MAR-B** (smallest of three; MAR-L 479M / MAR-H 943M too tight for 16GB at full quality)
- Protocol: **sampling-only with the official pretrained checkpoint**. No training.
- Sample count: **N=32** images (paper's headline FID-50K uses N=50000 — infeasible here)
- We do NOT claim to reproduce the paper's FID-50K = 2.31. We verify the architecture + sampling
  pipeline runs end-to-end and record real per-sample statistics.
- The paper's claimed FID is preserved as a reference value, not copied into measured.


## 1. Setup (clone repo + install)

In [ ]:
import os, time, json, sys, subprocess

t0 = time.time()

# Kaggle's default torch (~2.10+cu128) drops sm_60 (P100) kernels — Kaggle still
# allocates P100 to many accounts, so reinstall a wheel that includes sm_60 too.
# 2.4.1+cu121 covers sm_60..sm_90 inclusive; works on both P100 and T4.
subprocess.run(["pip", "install", "-q", "--upgrade",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

if not os.path.isdir("mar"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/LTH14/mar.git"], check=True)
os.chdir("mar")
sys.path.insert(0, os.getcwd())

subprocess.run(["pip", "install", "-q", "timm==0.9.12"], check=True)

import torch, numpy as np
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={gpu_name} sm={gpu_cap}")
print(f"setup elapsed: {time.time()-t0:.1f}s")


## 2. Download pretrained VAE + MAR-B checkpoints

In [ ]:
from util import download
t0 = time.time()
download.download_pretrained_vae(overwrite=False)
download.download_pretrained_marb(overwrite=False)
print(f"downloads elapsed: {time.time()-t0:.1f}s")

for p in ("pretrained_models/vae/kl16.ckpt",
          "pretrained_models/mar/mar_base/checkpoint-last.pth"):
    sz_mb = os.path.getsize(p) / (1024*1024) if os.path.exists(p) else 0
    print(f"  {p}: {sz_mb:.1f} MB")


## 3. Load MAR-B + VAE

In [ ]:
from models import mar
from models.vae import AutoencoderKL

torch.set_grad_enabled(False)
device = "cuda"

NUM_SAMPLING_STEPS_DIFFLOSS = 100
DIFFLOSS_D = 6
DIFFLOSS_W = 1024

t0 = time.time()
model = mar.__dict__["mar_base"](
    buffer_size=64,
    diffloss_d=DIFFLOSS_D,
    diffloss_w=DIFFLOSS_W,
    num_sampling_steps=str(NUM_SAMPLING_STEPS_DIFFLOSS),
).to(device)

state_dict = torch.load("pretrained_models/mar/mar_base/checkpoint-last.pth", map_location=device)["model_ema"]
model.load_state_dict(state_dict)
model.eval()

vae = AutoencoderKL(embed_dim=16, ch_mult=(1, 1, 2, 2, 4),
                    ckpt_path="pretrained_models/vae/kl16.ckpt").to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"MAR-B loaded: {n_params/1e6:.1f}M params (paper claims ~208M)")
print(f"load elapsed: {time.time()-t0:.1f}s")
print(f"GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. Sample N=32 images at 256×256

Reduced settings vs paper:
- N=32 (paper FID-50K uses 50000)
- num_iter=64 (paper uses 256 — 4× reduction)
- num_sampling_steps_diffloss=100 (paper value, kept)
- cfg=2.9, cfg_schedule=linear, temperature=1.0 (paper Table 5 MAR-B settings)
- 32 random ImageNet classes


In [ ]:
SEED = 42
NUM_SAMPLES = 32
NUM_AR_ITER = 64
CFG_SCALE = 2.9
CFG_SCHEDULE = "linear"
TEMPERATURE = 1.0

torch.manual_seed(SEED)
np.random.seed(SEED)

rng = np.random.RandomState(SEED)
class_labels = rng.randint(0, 1000, size=NUM_SAMPLES).tolist()

t_sample_start = time.time()
with torch.cuda.amp.autocast():
    sampled_tokens = model.sample_tokens(
        bsz=NUM_SAMPLES,
        num_iter=NUM_AR_ITER,
        cfg=CFG_SCALE,
        cfg_schedule=CFG_SCHEDULE,
        labels=torch.tensor(class_labels, device=device).long(),
        temperature=TEMPERATURE,
        progress=True,
    )
    sampled_images = vae.decode(sampled_tokens / 0.2325)
sampling_time_s = time.time() - t_sample_start

print(f"sampled {NUM_SAMPLES} images in {sampling_time_s:.1f}s ({sampling_time_s/NUM_SAMPLES:.2f}s per image)")
print(f"image tensor shape: {sampled_images.shape} dtype={sampled_images.dtype}")
print(f"GPU mem peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


## 5. Sanity stats on generated images

In [ ]:
imgs = sampled_images.float().clamp(-1, 1)
gen_pixel_mean = float(imgs.mean().item())
gen_pixel_std = float(imgs.std().item())
gen_pixel_min = float(imgs.min().item())
gen_pixel_max = float(imgs.max().item())

per_image_std = imgs.flatten(1).std(dim=1)
per_image_std_mean = float(per_image_std.mean().item())
per_image_std_min = float(per_image_std.min().item())

not_collapsed = bool(per_image_std_min > 0.05)

print(f"pixel range: [{gen_pixel_min:.3f}, {gen_pixel_max:.3f}]")
print(f"pixel mean / std: {gen_pixel_mean:.3f} / {gen_pixel_std:.3f}")
print(f"per-image std: mean={per_image_std_mean:.3f} min={per_image_std_min:.3f}")
print(f"not_collapsed (per_image_std_min > 0.05): {not_collapsed}")

from torchvision.utils import save_image
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
save_image(sampled_images, os.path.join(out_dir, "sample_grid.png"),
           nrow=8, normalize=True, value_range=(-1, 1))
print(f"saved grid: {out_dir}/sample_grid.png")


## 6. Write metrics.json

Honest reporting principles:
- **Do NOT** claim `fid_50k` — we did not measure it.
- **Do NOT** copy paper's claimed values into measured.
- **Do** report what was actually measured: pipeline verification + per-sample stats.
- Paper claim preserved as reference (separate file, never compared as if measured).


In [ ]:
measured = {
    "pipeline_verified": 1.0 if not_collapsed else 0.0,
    "model_params_m": float(n_params / 1e6),
    "samples_generated": float(NUM_SAMPLES),
    "sampling_time_s": float(sampling_time_s),
    "sampling_time_per_image_s": float(sampling_time_s / NUM_SAMPLES),
    "gpu_peak_gb": float(torch.cuda.max_memory_allocated() / 1e9),
    "num_ar_iter_used": float(NUM_AR_ITER),
    "num_ar_iter_paper": 256.0,
    "num_sampling_steps_diffloss_used": float(NUM_SAMPLING_STEPS_DIFFLOSS),
    "cfg_scale_used": float(CFG_SCALE),
    "gen_pixel_mean": gen_pixel_mean,
    "gen_pixel_std": gen_pixel_std,
    "per_image_std_mean": per_image_std_mean,
    "per_image_std_min": per_image_std_min,
}

paper_reference = {
    "model": "MAR-B",
    "fid_50k_paper": 2.31,
    "inception_score_paper": 281.7,
    "params_m_paper": 208.0,
    "note": "Paper FID-50K requires N=50000 sampling + ImageNet val moments. Out of scope for Kaggle T4 9h session at acceptable variance.",
}

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
out_path = os.path.join(out_dir, "metrics.json")
with open(out_path, "w") as f:
    json.dump(measured, f, indent=2)

ref_path = os.path.join(out_dir, "paper_reference.json")
with open(ref_path, "w") as f:
    json.dump(paper_reference, f, indent=2)

print(f"wrote metrics: {out_path}")
print(json.dumps(measured, indent=2))
print()
print(f"wrote reference: {ref_path}")
print(json.dumps(paper_reference, indent=2))


## Appendix — what this run does and does not show

**Shows:**
- The MAR-B pretrained checkpoint loads cleanly from the official Dropbox URL.
- End-to-end sampling pipeline (token sampling + VAE decode) runs on Kaggle T4 16GB without OOM.
- N=32 sampled images have non-degenerate per-image variance.
- Real wall-clock sampling time on T4 with paper's `num_sampling_steps=100`.

**Does NOT show:**
- Paper's headline FID-50K = 2.31 — N=50000 needed.
- Inception Score, Precision, Recall — same scale issue.
- MAR-L / MAR-H — too tight for T4.
- Any training — paper uses 32 H100s × 400 epochs.

**Expected verdict label:** `partial` — pipeline verified, headline metric NOT independently measured.
